# Setup and loading models from /models folder

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

model_names = ["RW", "DNS", "Ridge", "XGBoost"]

model_preds = {}
model_actuals = {}
model_metrics = {}

for name in model_names:
    path = MODEL_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded {name} from {path}, keys: {list(bundle.keys())}")

    model_preds[name]   = bundle["predictions"]
    model_actuals[name] = bundle["actuals"]
    model_metrics[name] = bundle["metrics"]   # <--- NEW

print("Models loaded:", list(model_preds.keys()))


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------- Paths -----------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# ----------------- Load raw FRED data -----------------
DGS1 = pd.read_csv(DATA_DIR / "DGS1.csv")
DGS2 = pd.read_csv(DATA_DIR / "DGS2.csv")
DGS5 = pd.read_csv(DATA_DIR / "DGS5.csv")
DGS10 = pd.read_csv(DATA_DIR / "DGS10.csv")

# 2y, 5y, 10y panel
merged = (
    DGS2.merge(DGS5, on="observation_date", how="inner")
        .merge(DGS10, on="observation_date", how="inner")
)
merged = merged.dropna(subset=["DGS2", "DGS5", "DGS10"])
merged = merged.set_index("observation_date")
merged.index = pd.to_datetime(merged.index, errors="coerce")

# Short rate (1y) as separate series
DGS1 = DGS1.dropna(subset=["DGS1"])
DGS1 = DGS1.set_index("observation_date")
DGS1.index = pd.to_datetime(DGS1.index, errors="coerce")

short_rate = DGS1["DGS1"]   # <-- THIS is what you pass to econ functions

# ----------------- Settings -----------------
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]
idx = merged.index
train_start = pd.Timestamp("2009-01-02")
train_end   = pd.Timestamp("2018-12-31")

test_start  = pd.Timestamp("2019-01-02")
test_end    = pd.Timestamp("2025-11-25")  

# Duration proxy (years-to-maturity)
maturity_durations = {
    "DGS2": 2.0,
    "DGS5": 5.0,
    "DGS10": 10.0
}



## Long–Short Trading Strategy

To assess whether yield forecasts translate into economic value, we implement a simple forecast-based long–short trading strategy following Carriero et al. (2012). The strategy is evaluated separately for each maturity (2-year, 5-year, and 10-year) and each forecast horizon  
\( h \in \{1,5,10,30\} \) trading days.

At each forecast date \(t\), the model produces a forecast of the excess holding-period return \( \widehat{er}_{t+h} \). A directional trading position is formed based solely on the sign of this forecast:

- \( \widehat{er}_{t+h} > 0 \): long position  
- \( \widehat{er}_{t+h} < 0 \): short position  
- \( \widehat{er}_{t+h} = 0 \): no position  

Formally, the trading signal is
\[
s_{t+h} = \operatorname{sign}(\widehat{er}_{t+h}).
\]

The realized excess holding-period return \( er_{t+h} \) is computed over the same horizon. Trading profits are then given by
\[
\pi_{t+h} = s_{t+h} \cdot er_{t+h} \cdot 100,
\]
where the constant factor represents a fixed notional investment and does not affect relative model comparisons.

Performance is evaluated over the out-of-sample period using the time series of trading profits. We report average trading profit, the standard deviation of profits, cumulative profit, and the Sharpe ratio, defined as the mean trading profit divided by its population standard deviation (not annualized). Sharpe ratios are interpreted as relative risk-adjusted performance measures and are comparable across models under the same trading rule.


In [ ]:
def compute_predicted_excess_return(pred_yield_series, merged_yields, short_rate,
                                    maturity_col, horizon, trading_days=252):
    D = maturity_durations[maturity_col]
    idx = pred_yield_series.index  # t+h

    y_start = merged_yields[maturity_col].shift(horizon).loc[idx]
    y_end_pred = pred_yield_series

    # convert yield change from percentage points to decimal
    dy_pred = (y_end_pred - y_start) / 100.0
    r_hat = -D * dy_pred

    # convert short rate (percent p.a.) to horizon return in decimal
    short_at_end = (short_rate.loc[idx] / 100.0) * (horizon / trading_days)

    er_pred = r_hat - short_at_end
    return er_pred.dropna()


def compute_realized_excess_return(merged_yields, short_rate, maturity_col,
                                   horizon, idx, trading_days=252):
    D = maturity_durations[maturity_col]
    y = merged_yields[maturity_col]

    y_end = y.loc[idx]
    y_start = y.shift(horizon).loc[idx]

    dy_real = (y_end - y_start) / 100.0
    r_real = -D * dy_real

    short_at_end = (short_rate.loc[idx] / 100.0) * (horizon / trading_days)

    er_real = r_real - short_at_end
    return er_real.dropna()

def sharpe_ratio(pi_series):
    """
    Sharpe = mean(π) / std(π), mit Populations-Std
    """
    mu = pi_series.mean()
    sigma = pi_series.std(ddof=0)
    return mu / sigma if sigma != 0 else np.nan

def compute_final_wealth(er_series, pos_series, h, W0=1000.0):
    """
    Compute final wealth from a long/short strategy with reinvestment.

    IMPORTANT:
    - er_series must be excess returns in DECIMALS (e.g. 0.001 = 0.1%).
    - pos_series is the position (+1, -1, or 0), aligned with er_series.
    - Uses NON-OVERLAPPING trades by taking every h-th observation to avoid
      double-counting overlapping h-day returns.
    """
    # Align and drop missing
    df = pd.DataFrame({"er": er_series, "pos": pos_series}).dropna()
    if df.empty:
        return np.nan

    # Non-overlapping sampling (critical!)
    df = df.iloc[::h].copy()
    if df.empty:
        return np.nan

    # Per-trade strategy return (decimal)
    strat_ret = df["pos"] * df["er"]

    # Safety: avoid impossible gross returns (would imply >100% loss in a trade)
    if (1.0 + strat_ret).le(0).any():
        return np.nan

    # Compound wealth
    final_wealth = W0 * (1.0 + strat_ret).prod()
    return float(final_wealth)




In [ ]:
# =====================================================
# Trading strategy: forecast-based long/short
# =====================================================

short_rate = DGS1["DGS1"]  
INITIAL_CAPITAL = 1000.0

idx = merged.index

model_names_trading = ["RW", "DNS", "Ridge", "XGBoost"]
model_pred_series = {name: model_preds[name] for name in model_names_trading}

# any model's actuals ok; using RW just as canonical key set
actual_series_dict = model_actuals["RW"]

trading_results = []

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        if key not in actual_series_dict:
            continue

        # Realized excess returns (same for all models)
        er_realized = compute_realized_excess_return(
            merged_yields=merged,
            short_rate=short_rate,
            maturity_col=maturity,
            idx=idx,
            horizon=h,
        )

        for model_name, pred_dict in model_pred_series.items():
            if key not in pred_dict:
                continue

            pred_yields = pred_dict[key]

            # Predicted excess returns for this model/maturity/horizon
            er_pred = compute_predicted_excess_return(
                pred_yield_series=pred_yields,
                merged_yields=merged,
                short_rate=short_rate,
                maturity_col=maturity,
                horizon=h,
            )

            # Align realized and predicted excess returns
            df_tmp = (
                pd.DataFrame({"er_realized": er_realized})
                .join(er_pred.rename("er_pred"), how="inner")
                .dropna()
            )
            if df_tmp.empty:
                continue

            # -----------------------------------------------------
            # Trading P&L (Carriero TR1):
            # π_{t+h} = sign(er̂_{t+h}) * er_{t+h} * 100
            # -----------------------------------------------------
            pnl = np.sign(df_tmp["er_pred"]) * df_tmp["er_realized"]

            avg_profit = pnl.mean()
            sr = sharpe_ratio(pnl)
            N = len(pnl)

            # -----------------------------------------------------
            # Hit ratio based on yield-change direction:
            # sign(Δy_pred) == sign(Δy_real)
            # -----------------------------------------------------
            eval_idx = df_tmp.index  # indexed by t+h

            y_start = merged[maturity].shift(h).loc[eval_idx]   # y_t
            y_end_real = merged[maturity].loc[eval_idx]         # y_{t+h}
            dy_real = (y_end_real - y_start)

            y_end_pred = pred_yields.loc[eval_idx]              # ŷ_{t+h}
            dy_pred = (y_end_pred - y_start)

            pos = np.sign(df_tmp["er_pred"])
            final_wealth = compute_final_wealth(df_tmp["er_realized"], pos, h=h, W0=1000.0)

            trading_results.append({
                "Maturity":   maturity,
                "Horizon":    h,
                "Model":      model_name,
                "Avg_Profit": avg_profit,
                "Sharpe":     sr,
                "N_Trades":   N,
                "Final_Wealth": final_wealth
            })


trading_results_df = (
    pd.DataFrame(trading_results)
    .sort_values(["Horizon", "Maturity", "Model"])
)
trading_results_df["Final_Wealth"] = trading_results_df["Final_Wealth"].round(2)

display(trading_results_df)


# Risk Reduction Strategy
To assess whether yield forecasts provide practical economic value, we implement a timing strategy aimed at reducing interest-rate risk. The investor holds an equal-weighted portfolio of zero-coupon U.S. Treasury bonds (2y, 5y, 10y) and we compare two approaches:

Buy-and-Hold (BH):
The investor remains fully invested throughout the entire out-of-sample period.

Forecast-Based Timing Strategy:
At each forecast horizon 
ℎ
∈
{
1
,
5
,
10
,
30
}
h∈{1,5,10,30}, the investment position depends on the sign of the forecasted excess return:

Position rule (plain text version):

If forecasted excess return > 0 → stay invested (position = 1)
If forecasted excess return ≤ 0 → move to cash (position = 0)


No leverage or short-selling is allowed.

Returns are measured as excess returns relative to the short-term rate, so holding cash corresponds to earning zero excess return.

Performance is evaluated using:

volatility

maximum drawdown

final wealth

A forecasting model is considered economically useful if it reduces drawdowns or improves risk-adjusted performance relative to Buy-and-Hold — especially in the rising-rate environment during 2019–2025.

In [ ]:
import numpy as np
import pandas as pd

TRADING_DAYS_PER_YEAR = 252

# Core configuration (only defined once in the notebook)
maturity_cols = ["DGS2", "DGS5", "DGS10"]
tau_years = np.array([2.0, 5.0, 10.0])  # in years


def compute_horizon_bond_returns(yields_dec, tau_years, h):
    """
    Approximate h-period holding returns for each maturity using
        r_{t,t+h}(τ) ≈ -τ (y_{t+h} - y_t),
    where yields are in decimal (not percent).

    Parameters
    ----------
    yields_dec : DataFrame
        DataFrame of yields in decimal form (e.g. 0.02, 0.05, ...)
        index = dates, columns = maturities (e.g. ['DGS2','DGS5','DGS10']).
    tau_years : array-like
        Vector of maturities in years, aligned with columns of yields_dec.
    h : int
        Horizon in trading days.

    Returns
    -------
    DataFrame
        Index = t (start dates), columns = maturities.
    """
    cols = list(yields_dec.columns)
    idx = yields_dec.index.to_list()

    rows = []
    dates = []

    for i in range(len(idx) - h):
        t   = idx[i]
        t_h = idx[i + h]

        y_t  = yields_dec.loc[t].values
        y_th = yields_dec.loc[t_h].values

        r_vec = -tau_years * (y_th - y_t)
        rows.append(r_vec)
        dates.append(t)

    r_df = pd.DataFrame(rows, index=pd.DatetimeIndex(dates), columns=cols)
    return r_df


def max_drawdown(return_series):
    """
    Computes max drawdown of the cumulative wealth path implied by returns.

    Parameters
    ----------
    return_series : Series
        Series of (excess) returns per period.

    Returns
    -------
    float
        Minimum drawdown (negative number).
    """
    cum = (1 + return_series).cumprod()
    peak = cum.cummax()
    dd = cum / peak - 1.0
    return dd.min()


In [ ]:
def _strategy_metrics(excess_series):
    """
    Compute mean, std, Sharpe, max drawdown, cumulative return,
    and final wealth for a given excess-return series.
    """
    mean = excess_series.mean()
    std = excess_series.std(ddof=1)
    sharpe = mean / std if std > 0 else np.nan
    mdd = max_drawdown(excess_series)
    cumret = (1 + excess_series).prod() - 1
    final_wealth = (1 + excess_series).cumprod().iloc[-1]
    return mean, std, sharpe, mdd, cumret, final_wealth


def run_portfolio_timing_overlay(
    model_name,
    pred_dict,
    merged,
    short_rate,
    maturity_cols=maturity_cols,
    tau_years=tau_years,
    horizons=None,
    weights=None,
    test_start=pd.Timestamp("2019-01-02"),
    test_end=pd.Timestamp("2025-11-25"),
):
    """
    Portfolio-timing overlay:
    - Base portfolio: long in all maturities with given weights.
    - Benchmark (BH): always invested (buy & hold).
    - Timing overlay: invest only when forecasted portfolio excess return > 0,
      otherwise stay in cash (no bond exposure, no leverage).

    Parameters
    ----------
    model_name : str
        Name of the forecasting model (for labeling).
    pred_dict : dict
        Dictionary {(maturity, horizon): forecast_yield_series}.
        This should come from model_preds[model_name].
    merged : DataFrame
        Yield panel with columns containing maturity_cols in percent (e.g. 2.0, 3.5).
    short_rate : Series
        Short rate (e.g. DGS1) in percent, indexed by date.
    maturity_cols : list
        List of maturity column names in merged.
    tau_years : array-like
        Vector of maturities in years, aligned with maturity_cols.
    horizons : list or None
        List of horizons in trading days. If None, defaults to [1, 5, 10, 30].
    weights : list or None
        Portfolio weights across maturities. If None, equal weights.
    test_start, test_end : Timestamp
        Out-of-sample evaluation window.

    Returns
    -------
    summary_df : DataFrame
        Summary metrics per horizon.
    detail_by_h : dict
        detail_by_h[h] -> DataFrame with columns ["BH_excess", "Timing_excess"].
    """
    if horizons is None:
        horizons = [1, 5, 10, 30]

    # Normalize portfolio weights
    if weights is None:
        weights = np.ones(len(maturity_cols)) / len(maturity_cols)
    else:
        weights = np.array(weights, dtype=float)
        weights = weights / weights.sum()

    results = []
    detail_by_h = {}

    # Yields in decimal once (DRY)
    yields_dec = merged[maturity_cols] / 100.0

    for h in horizons:
        print(f"\n=== {model_name} | horizon h={h} ===")

        # 1) Realized bond returns r_{t,t+h} for each maturity (decimal)
        r_all = compute_horizon_bond_returns(yields_dec, tau_years, h)
        # restrict to test window
        r_all = r_all.loc[(r_all.index >= test_start) & (r_all.index <= test_end)]

        # 2) Find common dates where we have forecasts for all maturities
        date_lists = []
        for col in maturity_cols:
            key = (col, h)
            if key not in pred_dict:
                date_lists = []
                break
            date_lists.append(pred_dict[key].index)

        if not date_lists:
            print(f"  -> No forecasts for horizon {h}, skipping.")
            continue

        common_dates = sorted(set(date_lists[0]).intersection(*date_lists[1:]))
        common_dates = [d for d in common_dates if (d >= test_start) and (d <= test_end)]

        # Intersection with dates where we have realized returns
        common_dates = sorted(set(common_dates).intersection(r_all.index))
        if not common_dates:
            print("  -> No common dates for forecasts and returns.")
            continue

        # Option: rebalance every h days (non-overlapping windows)
        rebalance_dates = common_dates[::h]

        bh_excess = []
        timing_excess = []

        for t in rebalance_dates:
            if t not in r_all.index:
                continue

            # realized bond return vector for t->t+h
            r_vec = r_all.loc[t].values  # shape (n_maturities,)
            r_port = np.dot(weights, r_vec)

            # risk-free h-period return from short_rate (DGS1)
            if t in short_rate.index:
                rf_t = short_rate.loc[t] / 100.0  # decimal
                rf_h = rf_t * (h / TRADING_DAYS_PER_YEAR)
            else:
                rf_h = 0.0

            # realized excess portfolio return
            er_port = r_port - rf_h

            # forecasts: y_t and y_hat_{t+h|t}
            if t not in yields_dec.index:
                continue
            y_t = yields_dec.loc[t].values  # actual yields at t (decimal)

            try:
                y_hat_vals = np.array(
                    [pred_dict[(col, h)].loc[t] for col in maturity_cols]
                ) / 100.0  # to decimal
            except KeyError:
                # missing forecast at t for some maturity
                continue

            # predicted bond returns per maturity
            r_hat_vec = -tau_years * (y_hat_vals - y_t)
            r_hat_port = np.dot(weights, r_hat_vec)
            er_hat_port = r_hat_port - rf_h  # expected excess portfolio return

            # Benchmark: always invested
            bh_excess.append((t, er_port))

            # Timing: invest only if expected excess return > 0
            position = 1.0 if er_hat_port > 0 else 0.0
            timing_excess.append((t, position * er_port))

        # 3) Build DataFrames
        if not bh_excess:
            print("  -> No valid observations, skipping.")
            continue

        bh_df = (
            pd.DataFrame(bh_excess, columns=["date", "BH_excess"])
            .set_index("date")
        )
        timing_df = (
            pd.DataFrame(timing_excess, columns=["date", "Timing_excess"])
            .set_index("date")
        )

        # align indexes
        idx_common = bh_df.index.intersection(timing_df.index)
        bh_df = bh_df.loc[idx_common]
        timing_df = timing_df.loc[idx_common]

        detail_by_h[h] = pd.concat([bh_df, timing_df], axis=1)

        # 4) Metrics (using helper to keep DRY)
        bh_series = detail_by_h[h]["BH_excess"]
        tm_series = detail_by_h[h]["Timing_excess"]
        N = len(bh_series)

        bh_mean, bh_std, bh_sharpe, bh_mdd, bh_cumret, bh_final_wealth = _strategy_metrics(bh_series)
        tm_mean, tm_std, tm_sharpe, tm_mdd, tm_cumret, tm_final_wealth = _strategy_metrics(tm_series)

        risk_reduction_std = (
            1.0 - (tm_std / bh_std) if bh_std > 0 and tm_std >= 0 else np.nan
        )

        print(f"  N={N}")
        print(
            f"  Buy&Hold    : mean={bh_mean:.6f}, std={bh_std:.6f}, "
            f"Sharpe={bh_sharpe:.3f}, MDD={bh_mdd:.3%}"
        )
        print(
            f"  Timing({model_name}): mean={tm_mean:.6f}, std={tm_std:.6f}, "
            f"Sharpe={tm_sharpe:.3f}, MDD={tm_mdd:.3%}"
        )
        print(f"  Std risk reduction: {risk_reduction_std:.3%}")

        results.append({
            "Model": model_name,
            "Horizon": h,
            #"N": N,
            #"BH_Mean": bh_mean,
            "BH_Std": bh_std,
            #"BH_Sharpe": bh_sharpe,
            "BH_MaxDD": bh_mdd,
            "BH_CumRet": bh_cumret,
            "BH_FinalWealth": bh_final_wealth,
            #"Timing_Mean": tm_mean,
            "Timing_Std": tm_std,
            #"Timing_Sharpe": tm_sharpe,
            "Timing_MaxDD": tm_mdd,
            #"Timing_CumRet": tm_cumret,
            "Timing_FinalWealth": tm_final_wealth,
            #"Std_Risk_Reduction": risk_reduction_std,
        })

    summary_df = pd.DataFrame(results)
    return summary_df, detail_by_h



In [ ]:
# Use the loaded model prediction dictionaries from the .pkl files
predictions = {
    "DNS":     model_preds["DNS"],
    "Ridge":   model_preds["Ridge"],
    "XGBoost": model_preds["XGBoost"],
    # optionally include RW if desired:
    # "RW": model_preds["RW"],
}

overlay_results = []
overlay_details = {}

for model_name, pred_dict in predictions.items():
    print(f"\n>>> Running timing overlay for {model_name}...")
    
    summary, details = run_portfolio_timing_overlay(
        model_name=model_name,
        pred_dict=pred_dict,
        merged=merged,
        short_rate=short_rate,
        horizons=[1,5,10,30],
        test_start=test_start,
        test_end=test_end
    )

    overlay_results.append(summary.assign(Model=model_name))
    overlay_details[model_name] = details

overlay_results_df = pd.concat(overlay_results, ignore_index=True)

overlay_results_df


# Robustness Check Risk-Reduction

In [ ]:
# =====================================================
# Robustness check: Risk-reduction (timing overlay)
# Half–half split with equal sample sizes (by # of rebalance dates)
# =====================================================

def _split_date_equal_count(dates, test_start, test_end):
    """Choose split date so that the count of dates in each half is (nearly) equal."""
    d = pd.DatetimeIndex(sorted(set(dates)))
    d = d[(d >= test_start) & (d <= test_end)]
    if len(d) < 10:
        raise ValueError("Not enough dates to split. Check your OOS window / inputs.")
    split = d[len(d) // 2]
    return split, d

def _compute_metrics_for_series(excess_series):
    """Return dict of metrics using your existing helper."""
    mean, std, sharpe, mdd, cumret, final_wealth = _strategy_metrics(excess_series.dropna())
    return {
        "Mean": mean,
        "Std": std,
        "Sharpe": sharpe,
        "MaxDD": mdd,
        "CumRet": cumret,
        "FinalWealth": final_wealth,
        "N": int(excess_series.dropna().shape[0]),
    }

def risk_reduction_robustness_table(
    model_names=("DNS", "Ridge", "XGBoost"),
    horizons=(1, 5, 10, 30),
    test_start=pd.Timestamp("2019-01-02"),
    test_end=pd.Timestamp("2025-11-25"),
    weights=None,
):
    """
    Runs the timing overlay for each model, then recomputes BH and Timing metrics
    separately for two equal-size subperiods.
    Returns:
      robustness_df, split_date
    """
    # 1) First run once for a "canonical" model to determine the equal-count split date
    # Use the first model in list to obtain rebalance dates for each horizon.
    canonical_model = list(model_names)[0]
    _, details_canon = run_portfolio_timing_overlay(
        model_name=canonical_model,
        pred_dict=model_preds[canonical_model],
        merged=merged,
        short_rate=short_rate,
        maturity_cols=maturity_cols,
        tau_years=tau_years,
        horizons=list(horizons),
        weights=weights,
        test_start=test_start,
        test_end=test_end,
    )

    # Build a unified set of evaluation dates (rebalance dates) across horizons
    # to choose a single split date applicable to all (simple & consistent).
    all_dates = []
    for h, df in details_canon.items():
        all_dates.extend(df.index.tolist())

    split_date, oos_dates = _split_date_equal_count(all_dates, test_start, test_end)
    print(
        f"Equal-count split date (risk-reduction): {split_date.date()}  "
        f"(N1={sum(oos_dates < split_date)}, N2={sum(oos_dates >= split_date)}, N_total={len(oos_dates)})"
    )

    # helpers for absolute deltas
    def std_reduction_abs(std_timing, std_bh):
        return np.nan if (std_bh is None or np.isnan(std_bh)) else (std_bh - std_timing)

    def mdd_reduction_abs(mdd_timing, mdd_bh):
        # MaxDD typically negative; reduction computed on absolute drawdown magnitude
        if mdd_timing is None or mdd_bh is None or np.isnan(mdd_timing) or np.isnan(mdd_bh):
            return np.nan
        return abs(mdd_bh) - abs(mdd_timing)

    def final_wealth_delta_abs(fw_timing, fw_bh):
        if fw_timing is None or fw_bh is None or np.isnan(fw_timing) or np.isnan(fw_bh):
            return np.nan
        return fw_timing - fw_bh

    # 2) Now run overlay for each model and compute subperiod metrics
    rows = []

    for model in model_names:
        _, detail_by_h = run_portfolio_timing_overlay(
            model_name=model,
            pred_dict=model_preds[model],
            merged=merged,
            short_rate=short_rate,
            maturity_cols=maturity_cols,
            tau_years=tau_years,
            horizons=list(horizons),
            weights=weights,
            test_start=test_start,
            test_end=test_end,
        )

        for h in horizons:
            if h not in detail_by_h:
                continue
            df = detail_by_h[h].dropna()

            # Subperiod masks based on the strategy decision dates (rebalance dates)
            mask_1 = df.index < split_date
            mask_2 = df.index >= split_date

            # Series
            bh_full = df["BH_excess"]
            tm_full = df["Timing_excess"]
            bh_1, tm_1 = bh_full.loc[mask_1], tm_full.loc[mask_1]
            bh_2, tm_2 = bh_full.loc[mask_2], tm_full.loc[mask_2]

            # Metrics dicts
            m_bh_1 = _compute_metrics_for_series(bh_1)
            m_tm_1 = _compute_metrics_for_series(tm_1)
            m_bh_2 = _compute_metrics_for_series(bh_2)
            m_tm_2 = _compute_metrics_for_series(tm_2)

            # Absolute risk reduction deltas (consistent with main section)
            rr_std_1 = std_reduction_abs(m_tm_1["Std"], m_bh_1["Std"])
            rr_std_2 = std_reduction_abs(m_tm_2["Std"], m_bh_2["Std"])

            rr_mdd_1 = mdd_reduction_abs(m_tm_1["MaxDD"], m_bh_1["MaxDD"])
            rr_mdd_2 = mdd_reduction_abs(m_tm_2["MaxDD"], m_bh_2["MaxDD"])

            fw_delta_1 = final_wealth_delta_abs(m_tm_1["FinalWealth"], m_bh_1["FinalWealth"])
            fw_delta_2 = final_wealth_delta_abs(m_tm_2["FinalWealth"], m_bh_2["FinalWealth"])

            rows.append({
                "Model": model,
                "Horizon": h,

                # First half
                "BH_Std_H1": m_bh_1["Std"],
                "BH_MaxDD_H1": m_bh_1["MaxDD"],
                "BH_FinalWealth_H1": m_bh_1["FinalWealth"],
                "Timing_Std_H1": m_tm_1["Std"],
                "Timing_FinalWealth_H1": m_tm_1["FinalWealth"],
                "Delta_Std_H1": rr_std_1,                    # absolute Δ Std
                "Delta_MaxDD_H1": rr_mdd_1,                  # absolute Δ MaxDD magnitude
                "Delta_FinalWealth_H1": fw_delta_1,          # absolute Δ FinalWealth

                # Second half
                "BH_Std_H2": m_bh_2["Std"],
                "BH_MaxDD_H2": m_bh_2["MaxDD"],
                "BH_FinalWealth_H2": m_bh_2["FinalWealth"],
                "Timing_Std_H2": m_tm_2["Std"],
                "Timing_FinalWealth_H2": m_tm_2["FinalWealth"],
                "Delta_Std_H2": rr_std_2,                    # absolute Δ Std
                "Delta_MaxDD_H2": rr_mdd_2,                  # absolute Δ MaxDD magnitude
                "Delta_FinalWealth_H2": fw_delta_2,          # absolute Δ FinalWealth
            })

    robustness_df = (
        pd.DataFrame(rows)
        .sort_values(["Horizon", "Model"])
        .reset_index(drop=True)
    )

    # rounding for readability
    for c in [col for col in robustness_df.columns if "Std" in col or "Delta_Std" in col]:
        robustness_df[c] = robustness_df[c].round(6)
    for c in [col for col in robustness_df.columns if "FinalWealth" in col]:
        robustness_df[c] = robustness_df[c].round(4)
    for c in [col for col in robustness_df.columns if "Delta_FinalWealth" in col]:
        robustness_df[c] = robustness_df[c].round(4)
    for c in [col for col in robustness_df.columns if "MaxDD" in col or "Delta_MaxDD" in col]:
        robustness_df[c] = robustness_df[c].round(4)

    return robustness_df, split_date


# ---- Run it ----
risk_robust_df, risk_split_date = risk_reduction_robustness_table(
    model_names=("DNS", "Ridge", "XGBoost"),
    horizons=horizons,
    test_start=test_start,
    test_end=test_end,
    weights=None,   # or pass your portfolio weights
)

display(risk_robust_df)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

H = 1              # horizon to plot
W0 = 1.0               # set to 1000.0 if you want wealth in euros

# --- 1) Run overlay for each model (only h=10) ---
_, detail_dns = run_portfolio_timing_overlay(
    model_name="DNS",
    pred_dict=model_preds["DNS"],
    merged=merged,
    short_rate=short_rate,
    horizons=[H],
)

_, detail_ridge = run_portfolio_timing_overlay(
    model_name="Ridge",
    pred_dict=model_preds["Ridge"],
    merged=merged,
    short_rate=short_rate,
    horizons=[H],
)

_, detail_xgb = run_portfolio_timing_overlay(
    model_name="XGBoost",
    pred_dict=model_preds["XGBoost"],
    merged=merged,
    short_rate=short_rate,
    horizons=[H],
)

# --- 2) Build a single aligned DataFrame of wealth curves ---
def wealth_from_excess(excess: pd.Series, W0: float = 1.0) -> pd.Series:
    return W0 * (1.0 + excess).cumprod()

# Use BH from DNS as the benchmark curve (same construction for this h)
bh_excess = detail_dns[H]["BH_excess"].sort_index()

# Align all timing series on a common date index (intersection)
idx_common = (
    bh_excess.index
    .intersection(detail_dns[H].index)
    .intersection(detail_ridge[H].index)
    .intersection(detail_xgb[H].index)
)

df = pd.DataFrame(index=idx_common)
df["Buy-and-Hold"] = wealth_from_excess(detail_dns[H].loc[idx_common, "BH_excess"], W0=W0)
df["DNS"]          = wealth_from_excess(detail_dns[H].loc[idx_common, "Timing_excess"], W0=W0)
df["Ridge"]        = wealth_from_excess(detail_ridge[H].loc[idx_common, "Timing_excess"], W0=W0)
df["XGBoost"]      = wealth_from_excess(detail_xgb[H].loc[idx_common, "Timing_excess"], W0=W0)

# --- 3) Plot ---
plt.figure(figsize=(12, 5))
for col in ["Buy-and-Hold", "DNS", "Ridge", "XGBoost"]:
    plt.plot(df.index, df[col], label=col)

#plt.title(f"Wealth Curves (Risk-Reduction Strategy, h={H}, OOS 2019–2025)")
plt.xlabel("Date")
plt.ylabel("Wealth Index" if W0 == 1.0 else "Wealth")
plt.legend()
plt.tight_layout()
plt.show()
